# Hurricane Otis (2023) — AI Model Track Comparison

Compare tropical cyclone tracks from **GraphCast**, **AIFS**, and **FCN3**
against ERA5 reanalysis for Hurricane Otis, at three lead times before
landfall (T−72 h, T−168 h, T−240 h).

**Verification target:** Otis landfall near Acapulco, Guerrero, Mexico,
snapped to `2023-10-25 06:00 UTC`.

| Lead  | Init time (UTC)  | Steps (6 h) |
|-------|------------------|-------------|
| 72 h  | 2023-10-22 06:00 | 12          |
| 168 h | 2023-10-18 06:00 | 28          |
| 240 h | 2023-10-15 06:00 | 40          |

In [ ]:
import os

os.environ["FORCE_CUDA_EXTENSION"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["EARTH2STUDIO_CACHE"] = os.path.expanduser("~/.cache/earth2studio")
os.environ["CUFILE_ENV_PATH_JSON"] = "/dev/null"

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm

from earth2studio.models.px import AIFS
from earth2studio.models.dx import TCTrackerWuDuan
from earth2studio.data import ARCO, CDS, fetch_data, prep_data_array
from earth2studio.utils.time import to_time_array
from earth2studio.utils.coords import map_coords

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
STORM_NAME        = "Otis"
VERIFICATION_TIME = datetime(2023, 10, 25, 6)
LEAD_TIMES_H      = [72, 168, 240]

INIT_TIMES = {h: VERIFICATION_TIME - timedelta(hours=h) for h in LEAD_TIMES_H}
N_STEPS    = {h: h // 6 for h in LEAD_TIMES_H}

OUT_DIR      = Path("./outputs/otis")
PLOT_EXTENT  = (-120, -85, 5, 25)
LANDFALL_LAT = 16.8
LANDFALL_LON = -99.9

MODEL_COLORS = {
    "AIFS":      "#2ca02c",
    "GraphCast": "#1f77b4",
    "ERA5":      "#000000",
}

OUT_DIR.mkdir(parents=True, exist_ok=True)

for h in LEAD_TIMES_H:
    print(f"  T-{h:>3}h : init={INIT_TIMES[h]}  steps={N_STEPS[h]}")

In [ ]:
def run_tc_inference(prognostic, data_source, start_time, nsteps, save_path):
    """Run a prognostic model through TCTrackerWuDuan and cache the track tensor."""
    if Path(save_path).exists():
        print(f"  cached → {save_path}")
        return torch.load(str(save_path), map_location="cpu")

    prognostic = prognostic.to(DEVICE)
    tracker = TCTrackerWuDuan()
    tracker.reset_path_buffer()

    x, coords = fetch_data(
        source=data_source,
        time=to_time_array([start_time]),
        variable=prognostic.input_coords()["variable"],
        lead_time=prognostic.input_coords()["lead_time"],
        device=DEVICE,
    )
    x, coords = map_coords(x, coords, prognostic.input_coords())

    iterator = prognostic.create_iterator(x, coords)
    for step, (x, coords) in tqdm(enumerate(iterator), total=nsteps + 1):
        x, coords = map_coords(x, coords, tracker.input_coords())
        output, _ = tracker(x, coords)
        output = output[:, 0]
        if step == nsteps:
            break

    tracks = output.cpu()
    torch.save(tracks, str(save_path))
    print(f"  saved  → {save_path}")
    return tracks

In [ ]:
def run_era5_truth(data_source, start_time, nsteps, save_path):
    """Track ERA5 reanalysis with TCTrackerWuDuan over the same time window as a forecast."""
    if Path(save_path).exists():
        print(f"  cached → {save_path}")
        return torch.load(str(save_path), map_location="cpu")

    tracker = TCTrackerWuDuan()
    tracker.reset_path_buffer()

    times = [start_time + timedelta(hours=6 * i) for i in range(nsteps + 1)]
    for time in tqdm(times):
        try:
            da = data_source(time, tracker.input_coords()["variable"])
        except RuntimeError:
            # Corrupted cache chunk — retry with a fresh uncached download.
            from earth2studio.data import ARCO
            da = ARCO(cache=False)(time, tracker.input_coords()["variable"])
        x, coords = prep_data_array(da, device=DEVICE)
        output, _ = tracker(x, coords)

    tracks = output.cpu()
    torch.save(tracks, str(save_path))
    print(f"  saved  → {save_path}")
    return tracks

In [ ]:
# ARCO provides ERA5 through Oct 2023; WB2ERA5 has a hard cutoff at 2023-01-11.
data_source = ARCO(cache=True)

In [ ]:
aifs_data_source = CDS(cache=True)

## AIFS forecasts

In [ ]:
%%time
aifs = AIFS.load_model(AIFS.load_default_package())

In [ ]:
%%time
aifs_tracks_72 = run_tc_inference(
    aifs, aifs_data_source,
    INIT_TIMES[72], N_STEPS[72],
    OUT_DIR / "aifs_72h.pt",
)

In [ ]:
%%time
aifs_tracks_168 = run_tc_inference(
    aifs, aifs_data_source,
    INIT_TIMES[168], N_STEPS[168],
    OUT_DIR / "aifs_168h.pt",
)

In [ ]:
%%time
aifs_tracks_240 = run_tc_inference(
    aifs, aifs_data_source,
    INIT_TIMES[240], N_STEPS[240],
    OUT_DIR / "aifs_240h.pt",
)

## ERA5 truth tracks

In [ ]:
%%time
era5_tracks_72 = run_era5_truth(
    data_source, INIT_TIMES[72], N_STEPS[72],
    OUT_DIR / "era5_72h.pt",
)

In [ ]:
%%time
era5_tracks_168 = run_era5_truth(
    data_source, INIT_TIMES[168], N_STEPS[168],
    OUT_DIR / "era5_168h.pt",
)

In [ ]:
%%time
era5_tracks_240 = run_era5_truth(
    data_source, INIT_TIMES[240], N_STEPS[240],
    OUT_DIR / "era5_240h.pt",
)

## Load saved tracks
Reload-safe: works whether you just ran inference or restarted the kernel.

In [ ]:
GRAPHCAST_TRACK_DIR = OUT_DIR  # override if GraphCast outputs live elsewhere

aifs_tracks_72  = torch.load(OUT_DIR / "aifs_72h.pt",  map_location="cpu")
aifs_tracks_168 = torch.load(OUT_DIR / "aifs_168h.pt", map_location="cpu")
aifs_tracks_240 = torch.load(OUT_DIR / "aifs_240h.pt", map_location="cpu")

era5_tracks_72  = torch.load(OUT_DIR / "era5_72h.pt",  map_location="cpu")
era5_tracks_168 = torch.load(OUT_DIR / "era5_168h.pt", map_location="cpu")
era5_tracks_240 = torch.load(OUT_DIR / "era5_240h.pt", map_location="cpu")

gc_tracks_72  = torch.load(GRAPHCAST_TRACK_DIR / "graphcast_72h.pt",  map_location="cpu")
gc_tracks_168 = torch.load(GRAPHCAST_TRACK_DIR / "graphcast_168h.pt", map_location="cpu")
gc_tracks_240 = torch.load(GRAPHCAST_TRACK_DIR / "graphcast_240h.pt", map_location="cpu")

print(f"AIFS  T-72h : {aifs_tracks_72.shape}")
print(f"ERA5  T-72h : {era5_tracks_72.shape}")
print(f"GCast T-72h : {gc_tracks_72.shape}")

## Track comparison maps

In [ ]:
def plot_tracks(models_list, extent, title, save_path=None):
    """Plot TC tracks on a Cartopy map. ERA5 solid, forecasts dashed; markers sized by 10-m wind."""
    fig, ax = plt.subplots(
        subplot_kw={"projection": ccrs.PlateCarree()},
        figsize=(11, 7),
    )

    ax.add_feature(cfeature.LAND,      facecolor="#e8e8e8", zorder=1)
    ax.add_feature(cfeature.OCEAN,     facecolor="#d0e8f5", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="0.3", zorder=3)
    ax.add_feature(cfeature.STATES,    linewidth=0.3, linestyle=":", edgecolor="0.5", zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.5, edgecolor="0.4", zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    for m in models_list:
        tracks = m["tracks"]
        if hasattr(tracks, "detach"):
            tracks = tracks.detach().cpu().numpy()

        ls = "-"  if m["name"].startswith("ERA5") else "--"
        lw = 2.2  if m["name"].startswith("ERA5") else 1.8
        label_used = False

        for p in range(tracks.shape[1]):
            lats = tracks[0, p, :, 0]
            lons = tracks[0, p, :, 1] - 360.0
            w10m = tracks[0, p, :, 3]
            mask = ~np.isnan(lats) & ~np.isnan(lons)
            if mask.sum() < 2:
                continue

            idx = np.where(mask)[0]
            lv, lnv, wv = lats[idx], lons[idx], w10m[idx]

            ax.plot(
                lnv, lv, color=m["color"], linewidth=lw, linestyle=ls,
                label=m["name"] if not label_used else "",
                transform=ccrs.PlateCarree(), zorder=5,
            )
            label_used = True

            sizes = np.clip(wv * 3.5, 15, 160)
            ax.scatter(
                lnv, lv, s=sizes, color=m["color"],
                edgecolors="white", linewidths=0.5,
                transform=ccrs.PlateCarree(), zorder=6,
            )

            for i, sz in [(0, 70), (-1, 90)]:
                ax.scatter(
                    lnv[i], lv[i], s=sz,
                    facecolors="none", edgecolors=m["color"], linewidths=1.6,
                    transform=ccrs.PlateCarree(), zorder=7,
                )

    ax.plot(
        LANDFALL_LON, LANDFALL_LAT,
        marker="*", color="#d62728", markersize=16,
        markeredgecolor="white", markeredgewidth=0.8,
        linestyle="None", label="Observed landfall",
        transform=ccrs.PlateCarree(), zorder=10,
    )

    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 9}
    gl.ylabel_style = {"size": 9}

    ax.legend(loc="lower right", fontsize=9, framealpha=0.9, edgecolor="0.7", handlelength=2.5)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    fig.tight_layout()

    if save_path:
        fig.savefig(str(save_path), dpi=150, bbox_inches="tight")
        print(f"saved → {save_path}")
    plt.show()

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Inputs in degrees, broadcasts with numpy."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def select_tc_path(track_tensor):
    """Return the path whose final position is closest to the known landfall point."""
    if hasattr(track_tensor, "numpy"):
        track_tensor = track_tensor.numpy()
    tracks = track_tensor[0]
    final_lats = tracks[:, -1, 0]
    final_lons = tracks[:, -1, 1]
    landfall_lon_360 = LANDFALL_LON % 360
    dists = haversine_km(final_lats, final_lons, LANDFALL_LAT, landfall_lon_360)
    return tracks[int(np.nanargmin(dists))]


def make_error_table(lead_h, models_dict):
    """Km-error table vs ERA5 truth; only rows where ERA5 has an active detection."""
    init_time  = INIT_TIMES[lead_h]
    era5_path  = select_tc_path(models_dict["ERA5 (truth)"])
    T          = era5_path.shape[0]

    model_paths = {
        name: select_tc_path(tensor)
        for name, tensor in models_dict.items()
        if name != "ERA5 (truth)"
    }

    rows = []
    for t in range(T):
        era_lat = era5_path[t, 0]
        era_lon = era5_path[t, 1]
        if np.isnan(era_lat):
            continue
        valid = init_time + timedelta(hours=6 * t)
        row = {"Valid (UTC)": valid.strftime("%m-%d %HZ")}
        for name, path in model_paths.items():
            m_lat = path[t, 0]
            m_lon = path[t, 1]
            km    = haversine_km(era_lat, era_lon, m_lat, m_lon)
            short = name.split()[0]
            row[f"{short} err (km)"] = "--" if np.isnan(km) else f"{km:.0f}"
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
plot_tracks(
    [
        {"name": "ERA5 (truth)", "tracks": era5_tracks_72, "color": MODEL_COLORS["ERA5"]},
        {"name": "AIFS",         "tracks": aifs_tracks_72, "color": MODEL_COLORS["AIFS"]},
        {"name": "GraphCast",    "tracks": gc_tracks_72,   "color": MODEL_COLORS["GraphCast"]},
    ],
    extent=PLOT_EXTENT,
    title=f"{STORM_NAME} (2023) — T−72 h forecast vs ERA5 truth (init {INIT_TIMES[72]:%Y-%m-%d %H}Z)",
    save_path=OUT_DIR / "track_72h.png",
)

In [ ]:
df_72 = make_error_table(72, {
    "ERA5 (truth)": era5_tracks_72,
    "AIFS":         aifs_tracks_72,
    "GraphCast":    gc_tracks_72,
})
print(f"\n=== T−72h position error (km) vs ERA5 truth ===\n")
print(df_72.to_string(index=False))

In [ ]:
plot_tracks(
    [
        {"name": "ERA5 (truth)", "tracks": era5_tracks_168, "color": MODEL_COLORS["ERA5"]},
        {"name": "AIFS",         "tracks": aifs_tracks_168, "color": MODEL_COLORS["AIFS"]},
        {"name": "GraphCast",    "tracks": gc_tracks_168,   "color": MODEL_COLORS["GraphCast"]},
    ],
    extent=PLOT_EXTENT,
    title=f"{STORM_NAME} (2023) — T−168 h forecast vs ERA5 truth (init {INIT_TIMES[168]:%Y-%m-%d %H}Z)",
    save_path=OUT_DIR / "track_168h.png",
)

In [ ]:
df_168 = make_error_table(168, {
    "ERA5 (truth)": era5_tracks_168,
    "AIFS":         aifs_tracks_168,
    "GraphCast":    gc_tracks_168,
})
print(f"\n=== T−168h position error (km) vs ERA5 truth ===\n")
print(df_168.to_string(index=False))

In [ ]:
plot_tracks(
    [
        {"name": "ERA5 (truth)", "tracks": era5_tracks_240, "color": MODEL_COLORS["ERA5"]},
        {"name": "AIFS",         "tracks": aifs_tracks_240, "color": MODEL_COLORS["AIFS"]},
        {"name": "GraphCast",    "tracks": gc_tracks_240,   "color": MODEL_COLORS["GraphCast"]},
    ],
    extent=PLOT_EXTENT,
    title=f"{STORM_NAME} (2023) — T−240 h forecast vs ERA5 truth (init {INIT_TIMES[240]:%Y-%m-%d %H}Z)",
    save_path=OUT_DIR / "track_240h.png",
)

In [ ]:
df_240 = make_error_table(240, {
    "ERA5 (truth)": era5_tracks_240,
    "AIFS":         aifs_tracks_240,
    "GraphCast":    gc_tracks_240,
})
print(f"\n=== T−240h position error (km) vs ERA5 truth ===\n")
print(df_240.to_string(index=False))

## Summary

**Hurricane Otis (2023)** — Eastern Pacific Category 5 landfall near Acapulco, Guerrero, MX.
One of the fastest-intensifying Atlantic/Pacific hurricanes on record; went from tropical storm
to Cat 5 in under 12 hours, making it a severe test of AI model track and intensity forecasting.

**Saved outputs in `./outputs/otis/`:**
- Forecast tracks: `aifs_72h.pt`, `aifs_168h.pt`, `aifs_240h.pt`
- ERA5 truth tracks: `era5_72h.pt`, `era5_168h.pt`, `era5_240h.pt`
- Track plots: `track_72h.png`, `track_168h.png`, `track_240h.png`

GraphCast `.pt` files are loaded from `GRAPHCAST_TRACK_DIR` (defaults to `OUT_DIR`);
override that variable if they were produced in a separate environment.